# 03. Export to Claude Desktop

`papers_scored.csv` 상위 N개를 markdown 표로 export → Claude Desktop 채팅에 붙여넣어 `reference_quality_check(rr)` 또는 `literature_synthesis(rr)` 프롬프트로 사용.

In [ ]:
!pip install -r ../../../requirements.txt

In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / 'papers_scored.csv')

TOP_N = 15
top = df.head(TOP_N).copy()
top.head()

,id,title,authors,year,venue,cited_by_count,citations_per_year,language,doi,oa_url,abstract,quality_score
0,https://openalex.org/W4360620450,Opinion Paper: “So what if ChatGPT wrote it?” ...,"Yogesh K. Dwivedi, Nir Kshetri, Laurie Hughes ...",2023,International Journal of Information Management,3446,1148.67,en,https://doi.org/10.1016/j.ijinfomgt.2023.102642,https://linkinghub.elsevier.com/science/articl...,"Transformative artificially intelligent tools,...",0.840
1,https://openalex.org/W4319662928,Performance of ChatGPT on USMLE: Potential for...,"Tiffany H. Kung, Morgan Cheatham, Arielle Mede...",2023,PLOS Digital Health,3435,1145.00,en,https://doi.org/10.1371/journal.pdig.0000198,https://journals.plos.org/digitalhealth/articl...,We evaluated the performance of a large langua...,0.828
2,https://openalex.org/W3140854437,"Review of deep learning: concepts, CNN archite...","Laith Alzubaidi, Jinglan Zhang, Amjad J. Humai...",2021,Journal Of Big Data,7319,1463.80,en,https://doi.org/10.1186/s40537-021-00444-8,https://journalofbigdata.springeropen.com/coun...,"In the last few years, the deep learning (DL) ...",0.803
3,https://openalex.org/W3035965352,Array programming with NumPy,"Charles R. Harris, K. Jarrod Millman, Stéfan J...",2020,Nature,21315,3552.50,en,https://doi.org/10.1038/s41586-020-2649-2,https://www.nature.com/articles/s41586-020-264...,"Array programming provides a powerful, compact...",0.800
4,https://openalex.org/W4411005431,Biomni: A General-Purpose Biomedical AI Agent,"Kexin Huang, Serena Zhang, Hanchen Wang 외 20명",2025,bioRxiv (Cold Spring Harbor Laboratory),65,65.00,en,https://doi.org/10.1101/2025.05.30.656746,https://www.biorxiv.org/content/biorxiv/early/...,Biomedical research underpins progress in our ...,0.799


In [3]:
import math


def _safe_str(value, default: str = '') -> str:
    """NaN/None/빈값 → default, 그 외에는 str(value).

    pandas의 NaN(float)이 "or" 폴백을 통과하지 않는 문제를 막는다.
    """
    if value is None:
        return default
    if isinstance(value, float) and math.isnan(value):
        return default
    s = str(value)
    return s if s else default


def _safe_int(value, default: int = 0) -> int:
    if value is None:
        return default
    if isinstance(value, float) and math.isnan(value):
        return default
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def to_markdown(df: 'pd.DataFrame') -> str:
    """상위 N개 논문을 Claude Desktop이 읽기 좋은 markdown 표 + abstract 묶음으로."""
    has_cpy = 'citations_per_year' in df.columns

    header = ['#', '제목', '저자', '연도', '인용수']
    if has_cpy:
        header.append('cit/yr')
    header += ['점수', 'DOI']

    lines = ['| ' + ' | '.join(header) + ' |',
             '|' + '|'.join(['---'] * len(header)) + '|']

    for i, row in df.reset_index(drop=True).iterrows():
        title = _safe_str(row.get('title')).replace('|', '\\|')[:120]
        authors = _safe_str(row.get('authors')).replace('|', '\\|')[:80]
        year = _safe_str(row.get('year'))
        cit = _safe_int(row.get('cited_by_count'))
        score = _safe_str(row.get('quality_score'))
        doi = _safe_str(row.get('doi'))

        cells = [str(i + 1), title, authors, year, str(cit)]
        if has_cpy:
            cells.append(_safe_str(row.get('citations_per_year')))
        cells += [score, doi]
        lines.append('| ' + ' | '.join(cells) + ' |')

    abstracts = ['', '## Abstracts', '']
    for i, row in df.reset_index(drop=True).iterrows():
        title = _safe_str(row.get('title'), default='(no title)')
        abs_text = _safe_str(row.get('abstract'), default='(no abstract)')
        abstracts.append(f'### [{i + 1}] {title}')
        abstracts.append(abs_text)
        abstracts.append('')

    return '\n'.join(lines + abstracts)


md = to_markdown(top)
out = DATA_DIR / f'papers_top_{TOP_N}.md'
out.write_text(md, encoding='utf-8')
print(f'Saved → {out.resolve()}')
print(f'Lines in output: {md.count(chr(10)) + 1}')
print('\n--- Preview ---\n')
print(md[:1200])


Saved → /Users/sungjae-cha/Documents/06 아름다운서당/next-seodang/projects/02_research_report_helper/data/papers_top_15.md
Lines in output: 65

--- Preview ---

| # | 제목 | 저자 | 연도 | 인용수 | cit/yr | 점수 | DOI |
|---|---|---|---|---|---|---|---|
| 1 | Opinion Paper: “So what if ChatGPT wrote it?” Multidisciplinary perspectives on opportunities, challenges and implicatio | Yogesh K. Dwivedi, Nir Kshetri, Laurie Hughes 외 70명 | 2023 | 3446 | 1148.67 | 0.84 | https://doi.org/10.1016/j.ijinfomgt.2023.102642 |
| 2 | Performance of ChatGPT on USMLE: Potential for AI-assisted medical education using large language models | Tiffany H. Kung, Morgan Cheatham, Arielle Medenilla 외 8명 | 2023 | 3435 | 1145.0 | 0.828 | https://doi.org/10.1371/journal.pdig.0000198 |
| 3 | Review of deep learning: concepts, CNN architectures, challenges, applications, future directions | Laith Alzubaidi, Jinglan Zhang, Amjad J. Humaidi 외 7명 | 2021 | 7319 | 1463.8 | 0.803 | https://doi.org/10.1186/s40537-021-00444-8 |
| 4

## 다음 단계

1. `data/papers_top_15.md` 파일을 열어 전체 내용을 복사.
2. Claude Desktop에서 `reference_quality_check(rr)` 프롬프트와 함께 붙여넣기.
3. 🟢 분류된 논문은 본문을 직접 PDF로 다운받아 읽기 (DOI 또는 OA URL 사용).
4. 본문에서 발견한 핵심을 한 줄씩 정리한 뒤 `literature_synthesis(rr)` 호출.